# Native Evaluation of Mistral-7B Financial NER Model

This notebook performs evaluation of the fine-tuned Mistral-7B model (`/kaggle/input/datasets/nam13092003/mistral-7b-financial-ner-final-lora-adapter`) using the native `evaluate.py` script from the `NER_finance` pipeline.

### Steps Covered:
1. **Environment Setup & Dependency Installation**
2. **Repository Alignment**: Clone or verify the `NER_finance` pipeline repo.
3. **Dataset Conversion**: Load character-based `ner_consolidated_vi.json` and convert it into the whitespace token-based **FIRE dataset format** expected by the pipeline's loader.
4. **Configuration Customization**: Inline modification of `default.yaml` configuration to point to the newly formatted Vietnamese test data and correct model configurations.
5. **Evaluation Execution**: Execute the pipeline's evaluation script using multi-GPU `accelerate launch`.

In [ ]:
# Step 1: Install Unsloth and pipeline dependencies
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install pyyaml pandas tqdm datasets

In [ ]:
# Step 2: Ensure pipeline repository is cloned in the Kaggle working directory
import os
if not os.path.exists("/kaggle/working/NER_finance"):
    print("Cloning NER_finance repository...")
    !git clone https://github.com/HaianCao/NER_finance.git /kaggle/working/NER_finance
else:
    print("NER_finance repository is already available.")

In [ ]:
# Step 3: Convert ner_consolidated_vi.json to FIRE format
import os
import json

DATASET_INPUT_PATH = "ner_consolidated_vi.json"
DATASET_OUTPUT_PATH = "/kaggle/working/ner_consolidated_vi_fire.json"

# Search recursively for input file if it is not in the current directory (useful on Kaggle)
actual_input_path = DATASET_INPUT_PATH
if not os.path.exists(actual_input_path):
    if os.path.exists(os.path.join("/kaggle/input", DATASET_INPUT_PATH)):
        actual_input_path = os.path.join("/kaggle/input", DATASET_INPUT_PATH)
    else:
        found = False
        for search_dir in [".", "/kaggle/input"]:
            if not os.path.exists(search_dir): continue
            for root, _, files in os.walk(search_dir):
                if os.path.basename(DATASET_INPUT_PATH) in files:
                    actual_input_path = os.path.join(root, os.path.basename(DATASET_INPUT_PATH))
                    found = True
                    break
            if found: break

def char_to_token_span(sentence: str, start_char: int, end_char: int):
    """
    Maps a character offset range [start_char, end_char) to whitespace-based word tokens.
    """
    words = sentence.split()
    word_spans = []
    current_idx = 0
    for w in words:
        start_pos = sentence.find(w, current_idx)
        if start_pos == -1:
            start_pos = current_idx
        end_pos = start_pos + len(w)
        word_spans.append((start_pos, end_pos))
        current_idx = end_pos
        
    overlapping_words = []
    for idx, (w_start, w_end) in enumerate(word_spans):
        if max(start_char, w_start) < min(end_char, w_end):
            overlapping_words.append(idx)
            
    if not overlapping_words:
        return -1, -1
        
    return overlapping_words[0], overlapping_words[-1]

print(f"Reading raw dataset from: {actual_input_path}")
with open(actual_input_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

converted_data = []
for idx, item in enumerate(raw_data):
    sentence = item.get("sentence", "")
    tokens = sentence.split()
    
    entities = []
    for ann_idx, ann in enumerate(item.get("annotations", [])):
        start_char = ann.get("start")
        end_char = ann.get("end")
        text = ann.get("text")
        label = ann.get("label")
        
        start_tok, end_tok = char_to_token_span(sentence, start_char, end_char)
        if start_tok == -1:
            continue
            
        # FIRE format: start token index (inclusive), end token index (exclusive)
        entities.append({
            "id": f"T{ann_idx}",
            "text": text,
            "type": label,
            "start": start_tok,
            "end": end_tok + 1
        })
        
    converted_data.append({
        "tokens": tokens,
        "entities": entities
    })

# Save the FIRE-formatted dataset
os.makedirs(os.path.dirname(DATASET_OUTPUT_PATH), exist_ok=True)
with open(DATASET_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=2)

print(f"Converted {len(converted_data)} samples to FIRE format and saved to: {DATASET_OUTPUT_PATH}")

In [ ]:
# Step 4: Load and modify the default pipeline configuration
import yaml

CONFIG_PATH = "/kaggle/working/NER_finance/configs/default.yaml"
MY_CONFIG = "/kaggle/working/my_config.yaml"

print(f"Loading config template from: {CONFIG_PATH}")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Update properties to evaluate our newly converted dataset using the Mistral-7b base architecture
cfg["training"]["format"] = "instruction"
cfg["model"]["name"] = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"

# Point to our converted FIRE file
cfg["data"]["test_files"] = [
    {"path": "/kaggle/working/ner_consolidated_vi_fire.json"}
]

# Save custom configuration
with open(MY_CONFIG, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print(f"Config saved to: {MY_CONFIG}")

## 5. Execute Evaluation using accelerate launch

The following block configures the accelerate default environment and executes the pipeline's evaluation script.

In [ ]:
# Configure defaults for accelerate and launch evaluation
CONFIG = "/kaggle/working/my_config.yaml"
CHECKPOINT = "/kaggle/input/datasets/nam13092003/mistral-7b-financial-ner-final-lora-adapter"

!accelerate config default

!accelerate launch \
    --multi_gpu \
    --num_processes=2 \
    /kaggle/working/NER_finance/evaluate.py \
    --config {CONFIG} \
    --checkpoint {CHECKPOINT} \
    --batch_size 10 \
    --output /kaggle/working/predictions.jsonl \
    2>&1 | tee /kaggle/working/eval_log.txt